# 02 · Un modelo que ve: VLM

Un VLM (modelo de visión y lenguaje) recibe una imagen junto con el texto. Se usa igual que el LLM del notebook anterior; lo único nuevo es que el mensaje lleva la imagen.

Qué vamos a ver:

1. Una imagen de prueba: una tarjeta de inspección.
2. Que el modelo la describa.
3. Que devuelva sus datos en JSON, con una estructura fija, y comprobar si acertó.
4. Lo mismo con una foto tuya.

In [ ]:
import os

from openai import OpenAI

BASE_URL = os.environ.get("VLM_BASE_URL") or os.environ["LLM_BASE_URL"]
API_KEY = os.environ.get("VLM_API_KEY", "local")
MODEL = os.environ["VLM_MODEL"]

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

print("Servidor:        ", BASE_URL)
print("Modelo de visión:", MODEL)

## 1. Una imagen de prueba

Para que el notebook funcione sin archivos externos, dibujamos la imagen aquí mismo: una tarjeta de inspección como las que se cuelgan en un equipo. Como la dibujamos nosotros, sabemos exactamente qué dice y podemos comprobar después si el modelo la leyó bien.

In [ ]:
from matplotlib import font_manager
from PIL import Image, ImageDraw, ImageFont

# Lo que dice la tarjeta. Cambia cualquier valor y vuelve a ejecutar.
CARD = {
    "equipo": "Compresor C-204",
    "linea": "L2",
    "turno": "nocturno",
    "presion_bar": 7.8,
    "temperatura_c": 92,
    "estado": "FUERA DE RANGO",
    "observacion": "Fuga de aceite en la válvula de descarga",
}


def font(size, bold=False):
    path = font_manager.findfont(font_manager.FontProperties(family="DejaVu Sans", weight="bold" if bold else "normal"))
    return ImageFont.truetype(path, size)


def draw_card(card):
    image = Image.new("RGB", (900, 560), "white")
    draw = ImageDraw.Draw(image)
    draw.rectangle([0, 0, 899, 559], outline="#111827", width=6)
    draw.rectangle([0, 0, 899, 90], fill="#094e39")
    draw.text((30, 24), "TARJETA DE INSPECCIÓN", font=font(38, bold=True), fill="white")

    lines = [
        ("Equipo", card["equipo"]),
        ("Línea", f"{card['linea']}        Turno: {card['turno']}"),
        ("Presión", f"{card['presion_bar']} bar"),
        ("Temperatura", f"{card['temperatura_c']} °C"),
        ("Estado", card["estado"]),
    ]
    y = 125
    for label, value in lines:
        draw.text((40, y), f"{label}:", font=font(30), fill="#4b5563")
        draw.text((270, y), value, font=font(30, bold=True), fill="#111827")
        y += 62

    # Luz de estado: roja si algo está fuera de rango.
    color = "#059669" if card["estado"].upper() == "OK" else "#dc2626"
    draw.ellipse([740, 330, 840, 430], fill=color, outline="#111827", width=4)

    draw.line([40, 450, 860, 450], fill="#d1d5db", width=2)
    draw.text((40, 470), "Observación:", font=font(26), fill="#4b5563")
    draw.text((40, 506), card["observacion"], font=font(26, bold=True), fill="#111827")
    return image


image = draw_card(CARD)
image

## 2. Describir la imagen

La imagen viaja dentro del mensaje, codificada en base64. El contenido del mensaje deja de ser un texto y pasa a ser una lista: una parte de texto y una parte de imagen.

In [ ]:
import base64
import io


def to_data_url(image):
    buffer = io.BytesIO()
    image.convert("RGB").save(buffer, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()


def ask_about(image, question, **options):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": question},
                    {"type": "image_url", "image_url": {"url": to_data_url(image)}},
                ],
            }
        ],
        **options,
    )
    return response.choices[0].message.content


print(ask_about(image, "Describe en español lo que ves en esta imagen, en pocas líneas."))

## 3. De la imagen a datos

Una descripción sirve para leerla; un sistema necesita datos. Con `response_format` se le da al modelo un esquema JSON y su respuesta sale siempre con esos campos y esos tipos, lista para guardarse en una base de datos o disparar una alerta.

In [ ]:
import json

SCHEMA = {
    "type": "object",
    "properties": {
        "equipo": {"type": "string"},
        "linea": {"type": "string"},
        "turno": {"type": "string"},
        "presion_bar": {"type": "number"},
        "temperatura_c": {"type": "number"},
        "estado": {"type": "string"},
        "observacion": {"type": "string"},
    },
    "required": ["equipo", "linea", "turno", "presion_bar", "temperatura_c", "estado", "observacion"],
}

answer = ask_about(
    image,
    "Extrae los datos de esta tarjeta de inspección. Copia los textos tal como aparecen.",
    response_format={"type": "json_schema", "json_schema": {"name": "inspeccion", "schema": SCHEMA}},
    temperature=0,
)

extracted = json.loads(answer)
extracted

¿Acertó? Comparamos campo por campo contra lo que dibujamos.

In [ ]:
import pandas as pd


def same(expected, got):
    if isinstance(expected, (int, float)):
        try:
            return abs(float(got) - float(expected)) < 0.05
        except (TypeError, ValueError):
            return False
    return str(expected).strip().lower() == str(got).strip().lower()


check = pd.DataFrame(
    [
        {
            "campo": field,
            "en la tarjeta": expected,
            "leyó el modelo": extracted.get(field),
            "correcto": "sí" if same(expected, extracted.get(field)) else "NO",
        }
        for field, expected in CARD.items()
    ]
)

correct = (check["correcto"] == "sí").sum()
print(f"{correct} de {len(check)} campos correctos")
check

Un modelo pequeño puede fallar en algún campo, sobre todo en textos largos o cifras. Vale la pena repetir este paso cuando esté descargado un modelo de visión más grande y comparar.

## 4. Con una foto tuya

Copia una foto a la carpeta `data/` del repositorio (aquí adentro se ve como `/workspace/data/`), escribe su nombre en `PHOTO` y ejecuta la celda. Puede ser una línea de producción, un tablero, una etiqueta.

In [ ]:
from pathlib import Path

PHOTO = ""  # por ejemplo: "mi_foto.jpg"

if PHOTO:
    photo = Image.open(Path("/workspace/data") / PHOTO)
    photo.thumbnail((1280, 1280))  # suficiente para el modelo y más rápido de enviar
    display(photo)
    print(ask_about(photo, "Describe lo que ves y lista los riesgos de seguridad que observes. Responde en español."))
else:
    print("Escribe el nombre de tu foto en PHOTO y vuelve a ejecutar esta celda.")